# MariaDB Data Warehouse Load

This notebook securely connects to MariaDB, loads the validated analytical datasets, and verifies database row counts.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env")

required_variables = [
    "DB_USER",
    "DB_PASSWORD",
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
]

missing_variables = [
    variable
    for variable in required_variables
    if not os.getenv(variable)
]

if missing_variables:
    raise ValueError(
        f"Missing environment variables: {missing_variables}"
    )

database_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(database_url, pool_pre_ping=True)

with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

    database_version = connection.execute(
        text("SELECT VERSION()")
    ).scalar()

print(f"Connected database: {database_name}")
print(f"MariaDB version: {database_version}")

Connected database: ecommerce_product_intelligence
MariaDB version: 12.3.3-MariaDB


## 1. Load Dimension Tables

The cleaned customer, seller, and product entities are loaded first because the transaction fact tables reference them.

In [2]:
from sqlalchemy import String, Integer, Float, Boolean

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

dimension_configs = {
    "dim_customers": {
        "file": "customers_clean.csv",
        "types": {
            "customer_id": String(32),
            "customer_unique_id": String(32),
            "customer_zip_code_prefix": Integer(),
            "customer_city": String(100),
            "customer_state": String(2),
            "customer_latitude": Float(),
            "customer_longitude": Float(),
            "customer_location_available": Boolean(),
        },
    },
    "dim_sellers": {
        "file": "sellers_clean.csv",
        "types": {
            "seller_id": String(32),
            "seller_zip_code_prefix": Integer(),
            "seller_city": String(100),
            "seller_state": String(2),
            "seller_latitude": Float(),
            "seller_longitude": Float(),
            "seller_location_available": Boolean(),
        },
    },
    "dim_products": {
        "file": "products_clean.csv",
        "types": {
            "product_id": String(32),
            "product_category_name": String(255),
            "product_category": String(255),
            "product_name_lenght": Float(),
            "product_description_lenght": Float(),
            "product_photos_qty": Float(),
            "product_weight_g": Float(),
            "product_length_cm": Float(),
            "product_height_cm": Float(),
            "product_width_cm": Float(),
            "category_was_missing": Boolean(),
            "product_weight_invalid": Boolean(),
        },
    },
}

dimension_frames = {}

for table_name, config in dimension_configs.items():
    dataframe = pd.read_csv(PROCESSED_DATA_DIR / config["file"])

    boolean_columns = [
        column
        for column in dataframe.columns
        if column.endswith("_available")
        or column.endswith("_missing")
        or column.endswith("_invalid")
    ]

    for column in boolean_columns:
        if dataframe[column].dtype == "object":
            dataframe[column] = dataframe[column].map({
                "True": True,
                "False": False,
                True: True,
                False: False,
            })

    sql_types = {
        column: sql_type
        for column, sql_type in config["types"].items()
        if column in dataframe.columns
    }

    dataframe.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1_000,
        method="multi",
        dtype=sql_types,
    )

    dimension_frames[table_name] = dataframe
    print(f"Loaded {table_name}: {len(dataframe):,} rows")

Loaded dim_customers: 99,441 rows
Loaded dim_sellers: 3,095 rows
Loaded dim_products: 32,951 rows


In [3]:
with engine.begin() as connection:
    connection.execute(
        text("ALTER TABLE dim_customers ADD PRIMARY KEY (customer_id)")
    )
    connection.execute(
        text("ALTER TABLE dim_sellers ADD PRIMARY KEY (seller_id)")
    )
    connection.execute(
        text("ALTER TABLE dim_products ADD PRIMARY KEY (product_id)")
    )

verification_records = []

with engine.connect() as connection:
    for table_name, dataframe in dimension_frames.items():
        sql_rows = connection.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar()

        verification_records.append({
            "table": table_name,
            "source_rows": len(dataframe),
            "sql_rows": sql_rows,
            "rows_match": len(dataframe) == sql_rows,
        })

dimension_verification = pd.DataFrame(verification_records)
dimension_verification

,table,source_rows,sql_rows,rows_match
0,dim_customers,99441,99441,True
1,dim_sellers,3095,3095,True
2,dim_products,32951,32951,True


## 2. Load the Order Fact Table

The order-level analytical table contains one row per order, combining customer, fulfillment, payment, review, and repeat-purchase metrics.

In [4]:
from sqlalchemy import Boolean, Date, DateTime, String, Text

order_fact_path = PROCESSED_DATA_DIR / "order_fact.csv"

order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "purchase_date",
    "purchase_month",
    "first_review_date",
    "latest_review_answer_timestamp",
]

available_columns = pd.read_csv(
    order_fact_path,
    nrows=0,
).columns.tolist()

order_fact_sql = pd.read_csv(
    order_fact_path,
    parse_dates=[
        column
        for column in order_date_columns
        if column in available_columns
    ],
    low_memory=False,
)

boolean_columns = [
    column
    for column in order_fact_sql.columns
    if column.startswith(("is_", "has_"))
    or column.endswith(("_available", "_outlier"))
    or column in ["timeline_anomaly", "payment_reconciled"]
]

for column in boolean_columns:
    order_fact_sql[column] = order_fact_sql[column].map({
        True: True,
        False: False,
        "True": True,
        "False": False,
    })

order_fact_types = {
    "order_id": String(32),
    "customer_id": String(32),
    "customer_unique_id": String(32),
    "order_status": String(20),
    "customer_city": String(100),
    "customer_state": String(2),
    "payment_methods": String(255),
    "primary_payment_method": String(30),
    "latest_review_title": Text(),
    "latest_review_message": Text(),
    "purchase_date": Date(),
    "order_purchase_timestamp": DateTime(),
    "order_approved_at": DateTime(),
    "order_delivered_carrier_date": DateTime(),
    "order_delivered_customer_date": DateTime(),
    "order_estimated_delivery_date": DateTime(),
    "purchase_month": DateTime(),
    "first_review_date": DateTime(),
    "latest_review_answer_timestamp": DateTime(),
}

for column in boolean_columns:
    order_fact_types[column] = Boolean()

order_fact_types = {
    column: sql_type
    for column, sql_type in order_fact_types.items()
    if column in order_fact_sql.columns
}

print("Loading fact_orders—this may take several minutes...")

order_fact_sql.to_sql(
    name="fact_orders",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi",
    dtype=order_fact_types,
)

print(f"Loaded fact_orders: {len(order_fact_sql):,} rows")

Loading fact_orders—this may take several minutes...
Loaded fact_orders: 99,441 rows


In [5]:
with engine.begin() as connection:
    connection.execute(text("""
        ALTER TABLE fact_orders
        ADD PRIMARY KEY (order_id),
        ADD INDEX idx_fact_orders_customer (customer_id),
        ADD INDEX idx_fact_orders_purchase_date (purchase_date),
        ADD INDEX idx_fact_orders_status (order_status),
        ADD CONSTRAINT fk_fact_orders_customer
            FOREIGN KEY (customer_id)
            REFERENCES dim_customers(customer_id)
    """))

with engine.connect() as connection:
    order_verification = connection.execute(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT order_id) AS unique_orders,
            MIN(purchase_date) AS earliest_purchase,
            MAX(purchase_date) AS latest_purchase,
            SUM(order_status = 'delivered') AS delivered_orders
        FROM fact_orders
    """)).mappings().one()

pd.DataFrame([order_verification])

,total_rows,unique_orders,earliest_purchase,latest_purchase,delivered_orders
0,99441,99441,2016-09-04,2018-10-17,96478


## 3. Load the Product Performance Fact Table

This table preserves one row per purchased item and connects product, seller, customer, geographic, fulfillment, and review information.

In [6]:
from sqlalchemy import Integer

item_fact_path = PROCESSED_DATA_DIR / "item_fact.csv"

item_date_columns = [
    "shipping_limit_date",
    "order_purchase_timestamp",
    "purchase_date",
    "purchase_month",
]

available_columns = pd.read_csv(
    item_fact_path,
    nrows=0,
).columns.tolist()

item_fact_sql = pd.read_csv(
    item_fact_path,
    parse_dates=[
        column
        for column in item_date_columns
        if column in available_columns
    ],
    low_memory=False,
)

boolean_columns = [
    column
    for column in item_fact_sql.columns
    if column.startswith(("is_", "has_"))
    or column.endswith(("_available", "_missing", "_invalid"))
]

for column in boolean_columns:
    item_fact_sql[column] = item_fact_sql[column].map({
        True: True,
        False: False,
        "True": True,
        "False": False,
    })

item_fact_types = {
    "order_id": String(32),
    "order_item_id": Integer(),
    "product_id": String(32),
    "seller_id": String(32),
    "customer_id": String(32),
    "customer_unique_id": String(32),
    "order_status": String(20),
    "product_category_name": String(255),
    "product_category": String(255),
    "seller_city": String(100),
    "seller_state": String(2),
    "customer_city": String(100),
    "customer_state": String(2),
    "shipping_limit_date": DateTime(),
    "order_purchase_timestamp": DateTime(),
    "purchase_date": Date(),
    "purchase_month": DateTime(),
}

for column in boolean_columns:
    item_fact_types[column] = Boolean()

item_fact_types = {
    column: sql_type
    for column, sql_type in item_fact_types.items()
    if column in item_fact_sql.columns
}

print("Loading fact_order_items—this may take several minutes...")

item_fact_sql.to_sql(
    name="fact_order_items",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi",
    dtype=item_fact_types,
)

print(f"Loaded fact_order_items: {len(item_fact_sql):,} rows")

Loading fact_order_items—this may take several minutes...
Loaded fact_order_items: 112,650 rows


In [11]:
schema_changes = [
    (
        "Primary key",
        """
        ALTER TABLE fact_order_items
        ADD PRIMARY KEY (order_id, order_item_id)
        """,
    ),
    (
        "Product index",
        """
        ALTER TABLE fact_order_items
        ADD INDEX idx_items_product (product_id)
        """,
    ),
    (
        "Seller index",
        """
        ALTER TABLE fact_order_items
        ADD INDEX idx_items_seller (seller_id)
        """,
    ),
    (
        "Purchase-date index",
        """
        ALTER TABLE fact_order_items
        ADD INDEX idx_items_purchase_date (purchase_date)
        """,
    ),
    (
        "Order foreign key",
        """
        ALTER TABLE fact_order_items
        ADD CONSTRAINT fk_items_order
        FOREIGN KEY (order_id)
        REFERENCES fact_orders(order_id)
        """,
    ),
    (
        "Product foreign key",
        """
        ALTER TABLE fact_order_items
        ADD CONSTRAINT fk_items_product
        FOREIGN KEY (product_id)
        REFERENCES dim_products(product_id)
        """,
    ),
    (
        "Seller foreign key",
        """
        ALTER TABLE fact_order_items
        ADD CONSTRAINT fk_items_seller
        FOREIGN KEY (seller_id)
        REFERENCES dim_sellers(seller_id)
        """,
    ),
]

for label, sql_statement in schema_changes:
    try:
        with engine.begin() as connection:
            connection.execute(text(sql_statement))
        print(f"PASSED: {label}")
    except Exception as error:
        original_error = getattr(error, "orig", error)
        print(f"FAILED: {label}")
        print(original_error)

FAILED: Primary key
(1068, 'Multiple primary key defined')
FAILED: Product index
(1061, "Duplicate key name 'idx_items_product'")
FAILED: Seller index
(1061, "Duplicate key name 'idx_items_seller'")
FAILED: Purchase-date index
(1061, "Duplicate key name 'idx_items_purchase_date'")
FAILED: Order foreign key
(1826, "Duplicate FOREIGN KEY constraint name ''")
FAILED: Product foreign key
(1826, "Duplicate FOREIGN KEY constraint name ''")
FAILED: Seller foreign key
(1826, "Duplicate FOREIGN KEY constraint name ''")


In [12]:
with engine.connect() as connection:
    constraints = pd.read_sql(text("""
        SELECT
            CONSTRAINT_NAME,
            CONSTRAINT_TYPE
        FROM information_schema.TABLE_CONSTRAINTS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'fact_order_items'
        ORDER BY CONSTRAINT_TYPE, CONSTRAINT_NAME
    """), connection)

    indexes = pd.read_sql(text("""
        SELECT DISTINCT
            INDEX_NAME
        FROM information_schema.STATISTICS
        WHERE TABLE_SCHEMA = DATABASE()
          AND TABLE_NAME = 'fact_order_items'
        ORDER BY INDEX_NAME
    """), connection)

print("Constraints:")
display(constraints)

print("Indexes:")
display(indexes)

Constraints:


,CONSTRAINT_NAME,CONSTRAINT_TYPE
0,fk_items_order,FOREIGN KEY
1,fk_items_product,FOREIGN KEY
2,fk_items_seller,FOREIGN KEY
3,PRIMARY,PRIMARY KEY


Indexes:


,INDEX_NAME
0,idx_items_product
1,idx_items_purchase_date
2,idx_items_seller
3,PRIMARY


In [13]:
with engine.connect() as connection:
    item_verification = pd.read_sql(text("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(
                DISTINCT CONCAT(order_id, '#', order_item_id)
            ) AS unique_item_keys,
            ROUND(SUM(price), 2) AS total_product_value,
            ROUND(SUM(freight_value), 2) AS total_freight_value,
            ROUND(
                100 * AVG(seller_customer_distance_km IS NOT NULL),
                2
            ) AS distance_coverage_percentage
        FROM fact_order_items
    """), connection)

item_verification

,total_rows,unique_item_keys,total_product_value,total_freight_value,distance_coverage_percentage
0,112650,112650,13591643.7,2251909.54,99.51


## 4. Validate the MariaDB Warehouse

Exact database row counts and foreign-key relationships are verified after loading.

In [14]:
warehouse_counts_query = text("""
    SELECT 'dim_customers' AS table_name, COUNT(*) AS database_rows
    FROM dim_customers

    UNION ALL

    SELECT 'dim_sellers', COUNT(*)
    FROM dim_sellers

    UNION ALL

    SELECT 'dim_products', COUNT(*)
    FROM dim_products

    UNION ALL

    SELECT 'fact_orders', COUNT(*)
    FROM fact_orders

    UNION ALL

    SELECT 'fact_order_items', COUNT(*)
    FROM fact_order_items
""")

with engine.connect() as connection:
    warehouse_counts = pd.read_sql(
        warehouse_counts_query,
        connection,
    )

expected_counts = {
    "dim_customers": 99_441,
    "dim_sellers": 3_095,
    "dim_products": 32_951,
    "fact_orders": 99_441,
    "fact_order_items": 112_650,
}

warehouse_counts["expected_rows"] = (
    warehouse_counts["table_name"].map(expected_counts)
)

warehouse_counts["rows_match"] = (
    warehouse_counts["database_rows"]
    == warehouse_counts["expected_rows"]
)

warehouse_counts

,table_name,database_rows,expected_rows,rows_match
0,dim_customers,99441,99441,True
1,dim_sellers,3095,3095,True
2,dim_products,32951,32951,True
3,fact_orders,99441,99441,True
4,fact_order_items,112650,112650,True


In [15]:
relationship_query = text("""
    SELECT
        (
            SELECT COUNT(*)
            FROM fact_orders AS orders
            LEFT JOIN dim_customers AS customers
                ON orders.customer_id = customers.customer_id
            WHERE customers.customer_id IS NULL
        ) AS orphan_order_customers,

        (
            SELECT COUNT(*)
            FROM fact_order_items AS items
            LEFT JOIN fact_orders AS orders
                ON items.order_id = orders.order_id
            WHERE orders.order_id IS NULL
        ) AS orphan_item_orders,

        (
            SELECT COUNT(*)
            FROM fact_order_items AS items
            LEFT JOIN dim_products AS products
                ON items.product_id = products.product_id
            WHERE products.product_id IS NULL
        ) AS orphan_item_products,

        (
            SELECT COUNT(*)
            FROM fact_order_items AS items
            LEFT JOIN dim_sellers AS sellers
                ON items.seller_id = sellers.seller_id
            WHERE sellers.seller_id IS NULL
        ) AS orphan_item_sellers
""")

with engine.connect() as connection:
    relationship_validation = pd.read_sql(
        relationship_query,
        connection,
    )

assert warehouse_counts["rows_match"].all()
assert (relationship_validation.iloc[0] == 0).all()

warehouse_counts.to_csv(
    PROJECT_ROOT / "reports" / "database_load_validation.csv",
    index=False,
)

print("MariaDB warehouse validation passed.")
relationship_validation

MariaDB warehouse validation passed.


,orphan_order_customers,orphan_item_orders,orphan_item_products,orphan_item_sellers
0,0,0,0,0
